# ACRS — Layer 1: hybrid ensemble

* a **logistic regression baseline**, so the hybrid ensemble is compared with a traditional scorecard-type model;
* training-set probabilities are saved (needed for PSI and calibration by period);
* best parameters saved to `best_params.json`.


In [2]:
import os, json, pickle, time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from sklearn.utils import resample
from scipy.stats import ks_2samp
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

OUT = 'acrs_data'
SEED = 42
N_TRIALS = int(os.environ.get('ACRS_N_TRIALS', 30))
TUNE_SAMPLE = 100_000

X_train = pd.read_parquet(f'{OUT}/X_train.parquet')
X_val = pd.read_parquet(f'{OUT}/X_val.parquet')
X_test = pd.read_parquet(f'{OUT}/X_test.parquet')
y_train = np.load(f'{OUT}/y_train.npy')
y_val = np.load(f'{OUT}/y_val.npy')
y_test = np.load(f'{OUT}/y_test.npy')

with open(f'{OUT}/feature_lists.json') as f:
    expected = json.load(f)['encoded_features']
assert list(X_train.columns) == expected, "Feature list differs from Phase 1 — re-run ACRS.ipynb"
BANNED = ['total_pymnt', 'total_rec', 'recoveries', 'last_pymnt', 'out_prncp', 'debt_settlement', 'hardship', 'pymnt_plan']
assert not [c for c in X_train.columns if any(b in c for b in BANNED)], "Leakage column found"
print(f"X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape} — {X_train.shape[1]} features, no leakage columns")

X_train (449414, 77) | X_val (373412, 77) | X_test (480812, 77) — 77 features, no leakage columns


## 1. Hyper-parameter tuning 

In [4]:
n_sample = min(TUNE_SAMPLE, len(X_train))
X_sample, y_sample = resample(X_train, y_train, n_samples=n_sample, random_state=SEED, stratify=y_train, replace=False)

def tune(name, make_model, space):
    def objective(trial):
        model = make_model(space(trial))
        model.fit(X_sample, y_sample)
        return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    t0 = time.time()
    study.optimize(objective, n_trials=N_TRIALS)
    print(f"{name:<14} best val ROC-AUC {study.best_value:.4f} in {time.time() - t0:.0f}s | {study.best_params}")
    return study

xgb_study = tune('XGBoost',
                 lambda p: XGBClassifier(**p, eval_metric='auc', random_state=SEED, n_jobs=-1, enable_categorical=False),
                 lambda t: {'n_estimators': t.suggest_int('n_estimators', 100, 500),
                            'max_depth': t.suggest_int('max_depth', 3, 8),
                            'learning_rate': t.suggest_float('learning_rate', 0.01, 0.2),
                            'subsample': t.suggest_float('subsample', 0.6, 1.0),
                            'colsample_bytree': t.suggest_float('colsample_bytree', 0.6, 1.0)})
lgbm_study = tune('LightGBM',
                  lambda p: LGBMClassifier(**p, random_state=SEED, n_jobs=-1, verbose=-1),
                  lambda t: {'n_estimators': t.suggest_int('n_estimators', 100, 500),
                             'max_depth': t.suggest_int('max_depth', 3, 8),
                             'learning_rate': t.suggest_float('learning_rate', 0.01, 0.2),
                             'subsample': t.suggest_float('subsample', 0.6, 1.0),
                             'colsample_bytree': t.suggest_float('colsample_bytree', 0.6, 1.0)})
rf_study = tune('Random Forest',
                lambda p: RandomForestClassifier(**p, random_state=SEED, n_jobs=-1),
                lambda t: {'n_estimators': t.suggest_int('n_estimators', 100, 400),
                           'max_depth': t.suggest_int('max_depth', 5, 15),
                           'max_features': t.suggest_categorical('max_features', ['sqrt', 'log2'])})

XGBoost        best val ROC-AUC 0.7379 in 119s | {'n_estimators': 394, 'max_depth': 3, 'learning_rate': 0.05687019004574091, 'subsample': 0.7222790607994186, 'colsample_bytree': 0.9303960651171188}
LightGBM       best val ROC-AUC 0.7371 in 89s | {'n_estimators': 425, 'max_depth': 3, 'learning_rate': 0.07477957039353499, 'subsample': 0.6158536464884385, 'colsample_bytree': 0.631845300449806}
Random Forest  best val ROC-AUC 0.7341 in 349s | {'n_estimators': 400, 'max_depth': 10, 'max_features': 'log2'}


## 2. Final models on the full training period, plus the logistic regression baseline

In [6]:
xgb_final = XGBClassifier(**xgb_study.best_params, eval_metric='auc', random_state=SEED, n_jobs=-1,
                          enable_categorical=False).fit(X_train, y_train)   # all features numeric; needed for SHAP
lgbm_final = LGBMClassifier(**lgbm_study.best_params, random_state=SEED, n_jobs=-1, verbose=-1).fit(X_train, y_train)
rf_final = RandomForestClassifier(**rf_study.best_params, random_state=SEED, n_jobs=-1).fit(X_train, y_train)
lr_final = LogisticRegression(max_iter=5000, C=1.0).fit(X_train, y_train)   # features are already min-max scaled
print("Trained: XGBoost, LightGBM, Random Forest (ensemble members) and logistic regression (baseline)")

Trained: XGBoost, LightGBM, Random Forest (ensemble members) and logistic regression (baseline)


## 3. Probabilities and headline metrics
The ensemble is the equal-weight average of XGBoost, LightGBM and Random Forest (soft voting, weight 1/3 each).
F1 is reported at the 0.50 threshold only for continuity; decisions use the thresholds chosen in Layer 3.

In [8]:
members = {'xgb': xgb_final, 'lgbm': lgbm_final, 'rf': rf_final}
data = {'train': (X_train, y_train), 'val': (X_val, y_val), 'test': (X_test, y_test)}
probs = {}
for split, (X, _) in data.items():
    for key, model in {**members, 'lr': lr_final}.items():
        probs[(key, split)] = model.predict_proba(X)[:, 1].astype('float64')
    probs[('ensemble', split)] = np.mean([probs[(k, split)] for k in members], axis=0)

def metrics(y, p):
    p = np.asarray(p, dtype='float64')
    auc = float(roc_auc_score(y, p))
    return {'roc_auc': round(auc, 4), 'ks': round(float(ks_2samp(p[y == 1], p[y == 0]).statistic), 4),
            'gini': round(2 * auc - 1, 4), 'brier': round(float(brier_score_loss(y, p)), 4),
            'f1_at_0.50': round(float(f1_score(y, (p >= 0.5).astype(int))), 4),
            'mean_pd': round(float(p.mean()), 4), 'default_rate': round(float(y.mean()), 4)}

NAMES = {'lr': 'Logistic regression', 'xgb': 'XGBoost', 'lgbm': 'LightGBM', 'rf': 'Random Forest', 'ensemble': 'Ensemble (L1)'}
results = {}
for split in ['val', 'test']:
    y = data[split][1]
    print(f"\n{split.upper()} ({len(y):,} loans, default rate {y.mean():.3f})")
    print(f"{'Model':<20} {'ROC-AUC':>8} {'KS':>7} {'Gini':>7} {'Brier':>7} {'F1@0.5':>7} {'mean PD':>8}")
    for key, name in NAMES.items():
        m = metrics(y, probs[(key, split)])
        results.setdefault(key, {})[split] = m
        print(f"{name:<20} {m['roc_auc']:>8.4f} {m['ks']:>7.4f} {m['gini']:>7.4f} {m['brier']:>7.4f} {m['f1_at_0.50']:>7.4f} {m['mean_pd']:>8.4f}")


VAL (373,412 loans, default rate 0.202)
Model                 ROC-AUC      KS    Gini   Brier  F1@0.5  mean PD
Logistic regression    0.7343  0.3425  0.4686  0.1433  0.1016   0.1723
XGBoost                0.7408  0.3512  0.4816  0.1416  0.0983   0.1752
LightGBM               0.7414  0.3524  0.4828  0.1416  0.1041   0.1746
Random Forest          0.7360  0.3420  0.4719  0.1451  0.0000   0.1673
Ensemble (L1)          0.7411  0.3514  0.4822  0.1423  0.0511   0.1723

TEST (480,812 loans, default rate 0.229)
Model                 ROC-AUC      KS    Gini   Brier  F1@0.5  mean PD
Logistic regression    0.7018  0.2934  0.4036  0.1645  0.1019   0.1744
XGBoost                0.7114  0.3068  0.4229  0.1627  0.0654   0.1707
LightGBM               0.7118  0.3064  0.4237  0.1627  0.0720   0.1704
Random Forest          0.7046  0.2932  0.4091  0.1654  0.0001   0.1677
Ensemble (L1)          0.7112  0.3049  0.4225  0.1633  0.0342   0.1696


## 4. Save models, probabilities and parameters

In [10]:
for key, model in {**members, 'lr': lr_final}.items():
    with open(f'{OUT}/{key}_final.pkl', 'wb') as f:
        pickle.dump(model, f)
for (key, split), p in probs.items():
    np.save(f'{OUT}/{key}_prob_{split}.npy', p)
best = {'xgb': xgb_study.best_params, 'lgbm': lgbm_study.best_params, 'rf': rf_study.best_params,
        'best_val_auc_tuning_sample': {'xgb': round(xgb_study.best_value, 4), 'lgbm': round(lgbm_study.best_value, 4),
                                       'rf': round(rf_study.best_value, 4)},
        'n_trials': N_TRIALS, 'tuning_sample': n_sample, 'seed': SEED,
        'lr': {'C': 1.0, 'max_iter': 5000}}
with open(f'{OUT}/best_params.json', 'w') as f:
    json.dump(best, f, indent=2)
with open(f'{OUT}/layer1_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved: *_final.pkl, *_prob_{train,val,test}.npy, best_params.json, layer1_metrics.json")
print("Layer 1 COMPLETE — run ACRS_Layer2.ipynb next")

Saved: *_final.pkl, *_prob_{train,val,test}.npy, best_params.json, layer1_metrics.json
Layer 1 COMPLETE — run ACRS_Layer2.ipynb next
